# Анализ гендерного распределения

Проанализируем статистики среди специальностей в разрезе доли женщин и качества работ.

Проверяемые статистические гипотезы:
1. `ShareWomen` и `Median` статистически значимо монотонно связаны
2. `ShareWomen` и `diploma_impact_rate` статистически значимо монотонно связаны
3. `ShareWomen` и `Unemployment_rate` статистически значимо монотонно связаны
4. `ShareWomen` и `FTR` (Full_time_year_round / Employed) статистически значимо монотонно связаны
5. `Total` и `diploma_impact_rate` статистически значимо монотонно связаны
6. `ShareWomen` значимо различается между категориями специальностей
7. Медианные зарплаты значимо различаются между специальностями с преобладанием мужчин (`ShareWomen` < 0.3) и женщин (`ShareWomen` > 0.7)

## Загрузка библиотек и установка настроек


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 50)


## Введение констант


In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")


## Загрузка набора данных


In [3]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "majors_recent_graduates_analytics.parquet")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Rank                  171 non-null    int64   
 1   Major_code            171 non-null    int64   
 2   Major                 171 non-null    category
 3   Total                 171 non-null    float64 
 4   Men                   171 non-null    float64 
 5   Women                 171 non-null    float64 
 6   Major_category        171 non-null    category
 7   ShareWomen            171 non-null    float64 
 8   Sample_size           171 non-null    int64   
 9   Employed              171 non-null    int64   
 10  Full_time             171 non-null    int64   
 11  Part_time             171 non-null    int64   
 12  Full_time_year_round  171 non-null    int64   
 13  Unemployed            171 non-null    int64   
 14  Unemployment_rate     171 non-null    float64 
 15  Median

## Гипотезы

### ShareWomen и Median статистически значимо монотонно связаны


$H_0$ — `ShareWomen` и `Median` не коррелируют;  
$H_1$ — между `ShareWomen` и `Median` существует монотонная связь.

In [5]:
x_col, y_col = "ShareWomen", "Median"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>Median vs ShareWomen</b>", font_size=13),
    xaxis_title="Доля женщин (ShareWomen)",
    yaxis_title="Медианная зарплата (Median)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Монотонность** — диаграмма рассеяния не показывает явной немонотонной зависимости.  
**Нормальность** — подходит для ненормального распределения.

Следовательно, критерий Спирмена применим.

#### Тест

In [6]:
rho, p_val = stats.spearmanr(df["ShareWomen"], df["Median"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = -0.6569, p = 0.0000


#### Размер эффекта

In [7]:
abs_rho = abs(rho)
if abs_rho < 0.1:
    magnitude = "тривиальный"
elif abs_rho < 0.3:
    magnitude = "слабый"
elif abs_rho < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта |ρ| = {abs_rho:.4f} ({magnitude})")


Размер эффекта |ρ| = 0.6569 (большой)


#### Вывод

Нулевая гипотеза об отсутствии корреляции отвергается: между `ShareWomen` и `Median` существует статистически значимая отрицательная связь (ρ = −0.66).  
Размер эффекта большой — специальности с более высокой долей женщин значительно ассоциированы с более низкими медианными зарплатами.

### ShareWomen и diploma_impact_rate статистически значимо монотонно связаны


$H_0$ — `ShareWomen` и `diploma_impact_rate` не коррелируют;  
$H_1$ — между `ShareWomen` и `diploma_impact_rate` существует монотонная связь.

In [8]:
x_col, y_col = "ShareWomen", "diploma_impact_rate"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>diploma_impact_rate vs ShareWomen</b>", font_size=13),
    xaxis_title="Доля женщин (ShareWomen)",
    yaxis_title="Доля трудоустройства по специальности (diploma_impact_rate)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Вывод

Данные не подтверждают гипотезу о зависимости между долей женщин и долей трудоустроенных по специальности.

### ShareWomen и Unemployment_rate статистически значимо монотонно связаны

$H_0$ — `ShareWomen` и `Unemployment_rate` не коррелируют;  
$H_1$ — между `ShareWomen` и `Unemployment_rate` существует монотонная связь.

In [11]:
x_col, y_col = "ShareWomen", "Unemployment_rate"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>Unemployment_rate vs ShareWomen</b>", font_size=13),
    xaxis_title="Доля женщин (ShareWomen)",
    yaxis_title="Уровень безработицы (Unemployment_rate)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Монотонность** — диаграмма рассеяния не показывает явной немонотонной зависимости.  
**Нормальность** — для долей (значения в [0, 1]) нормальность не гарантирована, поэтому предпочтителен ранговый критерий Спирмена.

Следовательно, критерий Спирмена применим.

#### Тест

In [12]:
rho, p_val = stats.spearmanr(df["ShareWomen"], df["Unemployment_rate"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = 0.0661, p = 0.3902


#### Размер эффекта

In [13]:
abs_rho = abs(rho)
if abs_rho < 0.1:
    magnitude = "тривиальный"
elif abs_rho < 0.3:
    magnitude = "слабый"
elif abs_rho < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта |ρ| = {abs_rho:.4f} ({magnitude})")


Размер эффекта |ρ| = 0.0661 (тривиальный)


#### Вывод

Нулевая гипотеза об отсутствии корреляции не отвергается (p = 0.390): связь между `ShareWomen` и `Unemployment_rate` статистически незначима.  
Данные не подтверждают гипотезу о зависимости между долей женщин и уровнем безработицы.

### ShareWomen и FTR статистически значимо монотонно связаны

$H_0$ — `ShareWomen` и `FTR` не коррелируют;  
$H_1$ — между `ShareWomen` и `FTR` существует монотонная связь.

In [14]:
x_col, y_col = "ShareWomen", "FTR"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>FTR vs ShareWomen</b>", font_size=13),
    xaxis_title="Доля женщин (ShareWomen)",
    yaxis_title="Доля занятых полный год (FTR)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Монотонность** — диаграмма рассеяния не показывает явной немонотонной зависимости.  
**Нормальность** — для долей (значения в [0, 1]) нормальность не гарантирована, поэтому предпочтителен ранговый критерий Спирмена.

Следовательно, критерий Спирмена применим.

#### Тест

In [15]:
rho, p_val = stats.spearmanr(df["ShareWomen"], df["FTR"])

print(f"Спирмен: ρ = {rho:.4f}, p = {p_val:.4f}")


Спирмен: ρ = -0.4043, p = 0.0000


#### Размер эффекта

In [16]:
abs_rho = abs(rho)
if abs_rho < 0.1:
    magnitude = "тривиальный"
elif abs_rho < 0.3:
    magnitude = "слабый"
elif abs_rho < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта |ρ| = {abs_rho:.4f} ({magnitude})")


Размер эффекта |ρ| = 0.4043 (средний)


#### Вывод

Нулевая гипотеза об отсутствии корреляции отвергается: между `ShareWomen` и `FTR` существует статистически значимая отрицательная связь (ρ = −0.40).  
Размер эффекта средний — специальности с более высокой долей женщин умеренно ассоциированы с более низкой долей занятых полный год.

### Total и diploma_impact_rate статистически значимо монотонно связаны


$H_0$ — `Total` и `diploma_impact_rate` не коррелируют;  
$H_1$ — между `Total` и `diploma_impact_rate` существует монотонная связь.

In [17]:
x_col, y_col = "Total", "diploma_impact_rate"
x = df[x_col]
y = df[y_col]
m, b = np.polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="#1f9e89", opacity=0.5, size=6),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=x_range, y=m * x_range + b,
    mode="lines",
    line=dict(color="#440154", width=1.5),
    showlegend=False,
))
fig.update_layout(
    title=dict(text="<b>diploma_impact_rate vs Total</b>", font_size=13),
    xaxis_title="Общее число выпускников (Total)",
    yaxis_title="Доля трудоустройства по специальности (diploma_impact_rate)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Вывод

 Данные не подтверждают гипотезу о зависимости между численностью выпускников и долей трудоустроенных по специальности.

### ShareWomen значимо различается между категориями специальностей


$H_0$ — распределения `ShareWomen` по группам `Major_category` не различаются;  
$H_1$ — хотя бы одна группа `Major_category` значимо отличается по `ShareWomen`.

In [20]:
order = (
    df.groupby("Major_category", observed=True)["ShareWomen"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

colors = pc.sample_colorscale("Viridis", len(order))
color_map = dict(zip(order, colors))

fig = px.box(
    df,
    y="Major_category",
    x="ShareWomen",
    color="Major_category",
    category_orders={"Major_category": order},
    color_discrete_map=color_map,
    labels={"ShareWomen": "Доля женщин", "Major_category": ""},
    title="<b>Доля женщин по категориям специальностей</b>",
    template="plotly_white",
)
fig.update_traces(
    boxpoints="outliers",
    marker=dict(size=4, opacity=0.5),
    line=dict(width=0.9),
    width=0.6,
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=900,
    height=580,
)
fig.show()


#### Предусловия для критерия Краскела–Уоллиса

**Независимость наблюдений** — каждое направление принадлежит ровно одной категории.  
**Тип данных** — `ShareWomen` является непрерывной переменной.  
**Форма распределений** — кардинально разных форм между группами не замечается.

Следовательно, критерий Краскела–Уоллиса применим.

#### Тест

In [21]:
groups = [g["ShareWomen"].values for _, g in df.groupby("Major_category", observed=True)]
h_stat, p_val = stats.kruskal(*groups)

print(f"Краскел–Уоллис: H = {h_stat:.4f}, p = {p_val}")


Краскел–Уоллис: H = 110.9058, p = 3.732838818565998e-17


#### Размер эффекта

In [22]:
n = sum(len(g) for g in groups)
k = len(groups)
eta_sq = (h_stat - k + 1) / (n - k)

print(f"Размер эффекта η² = {eta_sq:.4f}")


Размер эффекта η² = 0.6212


#### Вывод

Нулевая гипотеза об однородности распределений отвергается: `ShareWomen` статистически значимо различается между категориями специальностей.    
Размер эффекта η² = 0.62 подтверждает практическую значимость — доля женщин определяется категорией специальности в очень высокой степени.

### Медианные зарплаты значимо различаются между специальностями с преобладанием мужчин и женщин


$H_0$ — распределения `Median` в группах `ShareWomen < 0.3` и `ShareWomen > 0.7` не различаются;  
$H_1$ — распределения `Median` в группах `ShareWomen < 0.3` и `ShareWomen > 0.7` различаются.

In [23]:
male_dom = df[df["ShareWomen"] < 0.3]["Median"]
female_dom = df[df["ShareWomen"] > 0.7]["Median"]

fig = go.Figure()
for label, data, color in [
    (f"ShareWomen < 0.3 (n={len(male_dom)})", male_dom, "#440154"),
    (f"ShareWomen > 0.7 (n={len(female_dom)})", female_dom, "#1f9e89"),
]:
    fig.add_trace(go.Box(
        y=data,
        name=label,
        marker_color=color,
        boxpoints="outliers",
        marker=dict(size=4, opacity=0.5),
        line=dict(width=0.9),
    ))
fig.update_layout(
    title=dict(text="<b>Медианные зарплаты: специальности с преобладанием мужчин vs женщин</b>", font_size=13),
    yaxis_title="Медианная зарплата (Median)",
    template="plotly_white",
    width=700,
    height=500,
)
fig.show()


#### Предусловия для критерия Манна–Уитни

**Независимость наблюдений** — каждое направление принадлежит ровно одной группе.  
**Тип данных** — `Median` является непрерывной переменной.  
**Нормальность** — при малых размерах групп нормальность не гарантирована, поэтому предпочтителен непараметрический критерий Манна–Уитни.

Следовательно, критерий Манна–Уитни применим.

#### Тест

In [24]:
u_stat, p_val = stats.mannwhitneyu(male_dom, female_dom, alternative="two-sided")

print(f"Манн–Уитни: U = {u_stat:.0f}, p = {p_val:.4f}")
print(f"Медиана (ShareWomen < 0.3): {male_dom.median():,.0f}")
print(f"Медиана (ShareWomen > 0.7): {female_dom.median():,.0f}")


Манн–Уитни: U = 1413, p = 0.0000
Медиана (ShareWomen < 0.3): 50,000
Медиана (ShareWomen > 0.7): 33,000


#### Размер эффекта

In [25]:
n1, n2 = len(male_dom), len(female_dom)
r = 1 - 2 * u_stat / (n1 * n2)

abs_r = abs(r)
if abs_r < 0.1:
    magnitude = "тривиальный"
elif abs_r < 0.3:
    magnitude = "слабый"
elif abs_r < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Ранговая бисериальная корреляция:{r:.4f} ({magnitude})")


Ранговая бисериальная корреляция:-0.8890 (большой)


#### Вывод

Нулевая гипотеза об однородности медианных зарплат отвергается: зарплаты в специальностях с преобладанием мужчин (`ShareWomen` < 0.3) значимо выше, чем в специальностях с преобладанием женщин (`ShareWomen` > 0.7) — медианы 50 000 vs 33 000 ($).  
Размер эффекта большой (r = −0.89).

## Выводы

Проверяемые статистические гипотезы:

1. `ShareWomen` и `Median` статистически значимо монотонно связаны      
  -> Существует сильная отрицательная корреляция

2. `ShareWomen` и `diploma_impact_rate` статистически значимо монотонно связаны     
  -> Значимой корреляции не обнаружено

3. `ShareWomen` и `Unemployment_rate` статистически значимо монотонно связаны       
  -> Значимой корреляции не обнаружено

4. `ShareWomen` и `FTR` статистически значимо монотонно связаны     
  -> Существует умеренная отрицательная корреляция

5. `Total` и `diploma_impact_rate` статистически значимо монотонно связаны  
  -> Значимой корреляции не обнаружено

6. `ShareWomen` значимо различается между категориями специальностей        
  -> Категория специальности сильно определяет долю женщин

7. Медианные зарплаты значимо различаются между специальностями с преобладанием мужчин и женщин     
  -> Специальности с преобладанием мужчин имеют значимо более высокие зарплаты
